## **Arabic Named Entity Recognition**
Named entity recognition (NER) is the task of finding spans of text that constitute proper names and labeling the type of such entities.
As we discussed in the lecture, this task can be formulated as a sequence labeling problem, e.g., using the BIO tagging scheme.
Your task is to build a NER tagger for Arabic using Conditional Random Fields (CRFs) and explore different sets of features to achieve improvement over the baseline tagger.
Arabic is a morphologically rich language, which poses certain challenges to NLP.
We use ANERcorp that has 150K words annotated for four entities: Location (LOC), Organization (ORG), and Person (PER), and Miscellaneous (MISC).
Since ANERcorp only has train and test splits, we take 10% of the train as a development set.


## **The Task**
Your task is to improve the feature function (`word2features()`).
Currently, the baseline model only takes into account previous and next words.
You may use external tools to obtain useful features such as POS and morphological tags.
You will also need to submit a report describing your method, result, and analysis (1-2 pages).

## **Data Preparation**

In [1]:
%pip install scikit-learn seqeval
%pip install python-crfsuite


Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [3]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

base_dir = Path.cwd()
data_dir = base_dir / "ANERcorp-CamelLabSplits"
zip_path = base_dir / "ANERcorp-CamelLabSplits.zip"

if not all((data_dir / name).is_file() for name in (
    "ANERCorp_CamelLab_train.txt", "ANERCorp_CamelLab_test.txt"
)):
    if not zip_path.exists():
        urlretrieve(
            "https://camel.abudhabi.nyu.edu/anercorp/ANERcorp-CamelLabSplits.zip",
            zip_path,
        )
    with ZipFile(zip_path) as archive:
        for member in archive.infolist():
            if member.filename.startswith("ANERcorp-CamelLabSplits/"):
                archive.extract(member, base_dir)

print("Data directory:", data_dir)
print("Training file exists:", (data_dir / "ANERCorp_CamelLab_train.txt").is_file())
print("Test file exists:", (data_dir / "ANERCorp_CamelLab_test.txt").is_file())


Data directory: /Users/stella/Desktop/work/NLP_assignment1/ANERcorp-CamelLabSplits
Training file exists: True
Test file exists: True


In [4]:
from sklearn.model_selection import train_test_split

def read_data(file_path):
    with open(file_path, mode='r', encoding='utf-8') as f:
        sent, sents = [], []
        for line in f.readlines():
            if len(line.strip()) > 0:
                token = line.strip().split()
                word, ner = token
                sent.append((word, ner.replace('PERS', 'PER')))
            elif len(sent):
                sents.append(sent)
                sent = []
        if sent:
            sents.append(sent)
    return sents

_train_sents = read_data(data_dir / 'ANERCorp_CamelLab_train.txt')
test_sents = read_data(data_dir / 'ANERCorp_CamelLab_test.txt')

train_sents, dev_sents = train_test_split(_train_sents, test_size=0.1, random_state=12345)

## **Feature Engineering**
Here, your task is to design a function that tranforms each word into a set of features.
You may want to use existing tools to get POS, morphological features, etc.
The current model gives `0.70` in $F_1$ on the dev set.

In [6]:
import unicodedata

feature_sets = {
    "baseline": set(),
    "affixes": {"affixes"},
    "shape": {"shape"},
    "context": {"context"},
    "combined": {"affixes", "shape", "context"},
}


def word2features(sent, i, feature_set="baseline"):
    word = sent[i][0]
    enabled = feature_sets[feature_set]
    features = {"word": word}
    for offset in (-1, 1):
        j = i + offset
        if 0 <= j < len(sent):
            features[f"{offset:+d}:word"] = sent[j][0]
        else:
            features["BOS" if offset < 0 else "EOS"] = True

    if "affixes" in enabled:
        for size in (1, 2, 3):
            features[f"prefix{size}"] = word[:size]
            features[f"suffix{size}"] = word[-size:]

    if "shape" in enabled:
        features.update({
            "length": str(len(word)),
            "isdigit": word.isdigit(),
            "has_digit": any(c.isdigit() for c in word),
            "is_punct": all(unicodedata.category(c).startswith("P") for c in word),
            "has_latin": any("LATIN" in unicodedata.name(c, "") for c in word),
            "has_arabic": any("ARABIC" in unicodedata.name(c, "") for c in word),
        })

    if "context" in enabled:
        for offset in (-2, -1, 1, 2):
            j = i + offset
            if 0 <= j < len(sent):
                neighbor = sent[j][0]
                features[f"{offset:+d}:word"] = neighbor
                features[f"{offset:+d}:prefix2"] = neighbor[:2]
                features[f"{offset:+d}:suffix2"] = neighbor[-2:]
        if i > 0:
            features["prev+word"] = sent[i-1][0] + "|" + word
        if i + 1 < len(sent):
            features["word+next"] = word + "|" + sent[i+1][0]

    return features

Extract the features from the data and train your model.

In [7]:
import pycrfsuite

model_dir = base_dir / "models"
model_dir.mkdir(exist_ok=True)


def sent2features(sent, feature_set="baseline"):
    return [word2features(sent, i, feature_set) for i in range(len(sent))]


def sent2labels(sent):
    return [label for _, label in sent]


def train_model(sents, feature_set, model_path):
    trainer = pycrfsuite.Trainer(verbose=False)
    for sent in sents:
        trainer.append(sent2features(sent, feature_set), sent2labels(sent))
    trainer.set_params({
        "c1": 1.0,
        "c2": 1e-3,
        "max_iterations": 50,
        "feature.possible_transitions": True,
    })
    trainer.train(str(model_path))


def predict(sents, feature_set, model_path):
    tagger = pycrfsuite.Tagger()
    tagger.open(str(model_path))
    return [tagger.tag(sent2features(sent, feature_set)) for sent in sents]


for name in feature_sets:
    train_model(train_sents, name, model_dir / f"{name}.crfsuite")
    print(f"Trained {name}")

Trained baseline
Trained affixes
Trained shape
Trained context
Trained combined


## **Evaluation**

Evaluate your model on the development set and the test set.

**Note**: You may run evaluation on the test set **ONLY ONCE** at the end. You should not use the test set to check how well your model is currently performing and modify the model accordingly.
Use the dev set, or create a tune set for this purpose.

In [9]:
import csv
from seqeval.metrics import classification_report, precision_score, recall_score, f1_score


def evaluate(y_true, y_pred):
    return {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }


y_dev = [sent2labels(sent) for sent in dev_sents]
dev_predictions = {}
results = []
for name in feature_sets:
    predictions = predict(dev_sents, name, model_dir / f"{name}.crfsuite")
    dev_predictions[name] = predictions
    results.append({"features": name, **evaluate(y_dev, predictions)})

baseline_f1 = results[0]["f1"]
print(f"{'Features':<14} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Delta F1':>10}")
for row in results:
    row["delta_f1"] = row["f1"] - baseline_f1
    print(f"{row['features']:<14} {row['precision']:10.4f} {row['recall']:10.4f} {row['f1']:10.4f} {row['delta_f1']:+10.4f}")

with (base_dir / "dev_results.csv").open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(results[0]))
    writer.writeheader()
    writer.writerows(results)

Features        Precision     Recall         F1   Delta F1
baseline           0.8980     0.5967     0.7170    +0.0000
affixes            0.8534     0.6922     0.7644    +0.0475
shape              0.8821     0.5900     0.7071    -0.0099
context            0.8644     0.5878     0.6997    -0.0172
combined           0.8506     0.7022     0.7693    +0.0524


In [10]:
best_features = max(results, key=lambda row: row["f1"])["features"]
print("Selected features:", best_features)
print(classification_report(y_dev, dev_predictions[best_features], digits=4, zero_division=0))

Selected features: combined
              precision    recall  f1-score   support

         LOC     0.9052    0.7935    0.8456       397
        MISC     0.8269    0.4943    0.6187        87
         ORG     0.8062    0.5943    0.6842       175
         PER     0.7944    0.7054    0.7473       241

   micro avg     0.8506    0.7022    0.7693       900
   macro avg     0.8332    0.6468    0.7240       900
weighted avg     0.8487    0.7022    0.7660       900



In [ ]:
import json
import re

RUN_FINAL_TEST = True

if RUN_FINAL_TEST:
    prediction_path = base_dir / "ANERCorp_CamelLab_test.pred.txt"
    if prediction_path.exists():
        raise FileExistsError("Final test predictions already exist.")

    final_model = model_dir / f"{best_features}.crfsuite"
    y_pred_test = predict(test_sents, best_features, final_model)
    y_test = [sent2labels(sent) for sent in test_sents]
    test_metrics = evaluate(y_test, y_pred_test)
    test_report = classification_report(y_test, y_pred_test, digits=4, zero_division=0)

    source_lines = (data_dir / "ANERCorp_CamelLab_test.txt").read_text(
        encoding="utf-8"
    ).splitlines(keepends=True)
    flat_predictions = [label for sent in y_pred_test for label in sent]
    assert len(flat_predictions) == sum(bool(line.strip()) for line in source_lines)
    assert all(len(sent) == len(pred) for sent, pred in zip(test_sents, y_pred_test))
    labels = iter(flat_predictions)
    with prediction_path.open("x", encoding="utf-8") as f:
        for line in source_lines:
            if line.strip():
                label = next(labels)
                if label in {"B-PER", "I-PER"}:
                    label = label.replace("PER", "PERS")
                line = re.sub(r"\S+(\s*)$", lambda match: label + match.group(1), line)
            f.write(line)

    (base_dir / "test_results.json").write_text(
        json.dumps({
            "features": best_features,
            "training_scope": "90% of official training set; development set excluded",
            "training_sentences": len(train_sents),
            **test_metrics,
        }, indent=2), encoding="utf-8"
    )
    (base_dir / "test_report.txt").write_text(test_report, encoding="utf-8")
    print("Final features:", best_features)
    print("Training sentences:", len(train_sents))
    print(test_report)
    print("Predictions:", prediction_path)
